# Day 16 — Demand Prediction Dashboard

## EduPro Predictive Modeling

### Objective

Upgrade the Day 15 Streamlit prediction application
into an interactive business-oriented demand prediction dashboard.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
import joblib

In [2]:
from pathlib import Path

project_folder = Path(
    r"D:\Data Analytics Project\EduPro_Predictive_Modeling"
)

model_folder = project_folder / "models"
app_folder = project_folder / "app"
notebook_folder = project_folder / "notebooks"

day14_file_path = (
    project_folder
    / "data"
    / "model_data"
    / "EduPro_Day14_Final_Model_Predictions.xlsx"
)

enrollment_model_path = (
    model_folder
    / "EduPro_Final_Enrollment_Model.joblib"
)

revenue_model_path = (
    model_folder
    / "EduPro_Final_Revenue_Model.joblib"
)

app_file_path = app_folder / "app.py"

notebook_folder.mkdir(
    parents=True,
    exist_ok=True
)

print("Project:", project_folder)
print("App:", app_file_path)
print("Enrollment model:", enrollment_model_path)
print("Revenue model:", revenue_model_path)

Project: D:\Data Analytics Project\EduPro_Predictive_Modeling
App: D:\Data Analytics Project\EduPro_Predictive_Modeling\app\app.py
Enrollment model: D:\Data Analytics Project\EduPro_Predictive_Modeling\models\EduPro_Final_Enrollment_Model.joblib
Revenue model: D:\Data Analytics Project\EduPro_Predictive_Modeling\models\EduPro_Final_Revenue_Model.joblib


## Validate Day 15 artifacts

In [3]:
required_files = {
    "Enrollment model": enrollment_model_path,
    "Revenue model": revenue_model_path,
    "Day 14 output": day14_file_path,
    "Streamlit app": app_file_path
}

for name, path in required_files.items():
    print(f"{name}: {path.exists()}")

    if not path.exists():
        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

print("\nAll required Day 16 artifacts are available.")

Enrollment model: True
Revenue model: True
Day 14 output: True
Streamlit app: True

All required Day 16 artifacts are available.


## Load models

In [19]:
enrollment_model = joblib.load(
    enrollment_model_path
)

revenue_model = joblib.load(
    revenue_model_path
)

print("Enrollment model:", type(enrollment_model))
print("Revenue model:", type(revenue_model))

Enrollment model: <class 'sklearn.pipeline.Pipeline'>
Revenue model: <class 'sklearn.pipeline.Pipeline'>


## Define feature lists

In [20]:
original_features = [
    "CourseCategory",
    "CourseType",
    "CourseLevel",
    "CoursePrice",
    "CourseDuration",
    "CourseRating",
    "TeacherRating",
    "YearsOfExperience",
    "Expertise"
]

engineered_features = [
    "PricePerDay",
    "PriceSquared",
    "DurationSquared",
    "LogCoursePrice",
    "LogCourseDuration",
    "RatingGap",
    "AverageRating",
    "CourseQualityScore",
    "ExperienceRatingScore",
    "PriceRatingInteraction",
    "PricePerRatingPoint",
    "Category_Type",
    "Category_Level",
    "Type_Level",
    "Category_Expertise",
    "Level_Expertise",
    "ExperienceBand"
]

all_modeling_features = (
    original_features +
    engineered_features
)

print(len(original_features))
print(len(engineered_features))
print(len(all_modeling_features))

9
17
26


## Reuse Day 13 Feature Engineering

In [24]:
# ============================================================
# DAY 16 — RECREATE DAY 13 FEATURE ENGINEERING
# ============================================================

def create_engineered_features(df):

    data = df.copy()

    # --------------------------------------------------------
    # Price / duration features
    # --------------------------------------------------------

    data["PricePerDay"] = (
        data["CoursePrice"]
        /
        data["CourseDuration"].replace(0, np.nan)
    )

    data["PricePerDay"] = (
        data["PricePerDay"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    data["PriceSquared"] = (
        data["CoursePrice"] ** 2
    )

    data["DurationSquared"] = (
        data["CourseDuration"] ** 2
    )

    data["LogCoursePrice"] = np.log1p(
        data["CoursePrice"]
    )

    data["LogCourseDuration"] = np.log1p(
        data["CourseDuration"]
    )

    # --------------------------------------------------------
    # Quality / rating features
    # --------------------------------------------------------

    data["RatingGap"] = (
        data["CourseRating"]
        -
        data["TeacherRating"]
    )

    data["AverageRating"] = (
        data["CourseRating"]
        +
        data["TeacherRating"]
    ) / 2

    data["CourseQualityScore"] = (
        data["CourseRating"]
        *
        data["TeacherRating"]
    )

    data["ExperienceRatingScore"] = (
        data["YearsOfExperience"]
        *
        data["TeacherRating"]
    )

    # --------------------------------------------------------
    # Price / quality interaction
    # --------------------------------------------------------

    data["PriceRatingInteraction"] = (
        data["CoursePrice"]
        *
        data["AverageRating"]
    )

    data["PricePerRatingPoint"] = (
        data["CoursePrice"]
        /
        data["AverageRating"].replace(0, np.nan)
    )

    data["PricePerRatingPoint"] = (
        data["PricePerRatingPoint"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    # --------------------------------------------------------
    # Categorical interaction features
    # --------------------------------------------------------

    data["Category_Type"] = (
        data["CourseCategory"].astype(str)
        + "_"
        + data["CourseType"].astype(str)
    )

    data["Category_Level"] = (
        data["CourseCategory"].astype(str)
        + "_"
        + data["CourseLevel"].astype(str)
    )

    data["Type_Level"] = (
        data["CourseType"].astype(str)
        + "_"
        + data["CourseLevel"].astype(str)
    )

    data["Category_Expertise"] = (
        data["CourseCategory"].astype(str)
        + "_"
        + data["Expertise"].astype(str)
    )

    data["Level_Expertise"] = (
        data["CourseLevel"].astype(str)
        + "_"
        + data["Expertise"].astype(str)
    )

    # --------------------------------------------------------
    # Experience band
    # --------------------------------------------------------

    data["ExperienceBand"] = pd.cut(
        data["YearsOfExperience"],
        bins=[
            -np.inf,
            5,
            10,
            20,
            np.inf
        ],
        labels=[
            "Early",
            "Developing",
            "Experienced",
            "Highly_Experienced"
        ]
    )

    return data


print("Day 13 feature engineering function recreated successfully.")

Day 13 feature engineering function recreated successfully.


## Create Prediction function

In [21]:
def predict_course(course_input):

    revenue_input = course_input[
        original_features
    ].copy()

    engineered_input = create_engineered_features(
        course_input
    )

    enrollment_prediction = (
        enrollment_model.predict(
            engineered_input[
                all_modeling_features
            ]
        )[0]
    )

    revenue_prediction = (
        revenue_model.predict(
            revenue_input
        )[0]
    )

    enrollment_prediction = max(
        0,
        enrollment_prediction
    )

    revenue_prediction = max(
        0,
        revenue_prediction
    )

    return {
        "EnrollmentCount": enrollment_prediction,
        "CourseRevenue": revenue_prediction
    }

## Demand interpretation function

In [22]:
def classify_demand(enrollment):

    if enrollment < 150:
        return "Low"

    elif enrollment < 170:
        return "Moderate"

    elif enrollment < 185:
        return "High"

    else:
        return "Very High"

## Business Recommendation Function

In [23]:
def generate_recommendation(demand_level):

    if demand_level == "Low":
        return (
            "Review course pricing, positioning, "
            "target audience and promotional strategy."
        )

    elif demand_level == "Moderate":
        return (
            "Consider launching with targeted "
            "marketing and demand monitoring."
        )

    elif demand_level == "High":
        return (
            "Strong candidate for launch. "
            "Consider instructor and capacity planning."
        )

    else:
        return (
            "Very strong predicted demand. "
            "Prioritize launch and prepare sufficient capacity."
        )

## Test the Dashboard Prediction Logic

In [25]:
test_course = pd.DataFrame([
    {
        "CourseCategory": "Technology",
        "CourseType": "Online",
        "CourseLevel": "Intermediate",
        "CoursePrice": 490.9,
        "CourseDuration": 7.55,
        "CourseRating": 4.55,
        "TeacherRating": 4.58,
        "YearsOfExperience": 24,
        "Expertise": "Programming"
    }
])

prediction = predict_course(test_course)

demand_level = classify_demand(
    prediction["EnrollmentCount"]
)

recommendation = generate_recommendation(
    demand_level
)

print("Predicted Enrollment:")
print(round(prediction["EnrollmentCount"], 2))

print("\nPredicted Revenue:")
print(round(prediction["CourseRevenue"], 2))

print("\nDemand Level:")
print(demand_level)

print("\nRecommendation:")
print(recommendation)

Predicted Enrollment:
168.23

Predicted Revenue:
82525.93

Demand Level:
Moderate

Recommendation:
Consider launching with targeted marketing and demand monitoring.


# 24. Cell 12 — Dashboard Design Specification

## Dashboard Components

The Day 16 Demand Prediction Dashboard will extend the Day 15 Streamlit prediction app by adding business-oriented prediction interpretation and recommendation features.

### 1. Header
- Display the EduPro Demand Prediction Dashboard title.
- Explain that the dashboard predicts course enrollment demand and revenue.

### 2. Course Input Section
- Allow the user to enter/select course characteristics:
  - Course Category
  - Course Type
  - Course Level
  - Course Price
  - Course Duration
  - Course Rating
  - Teacher Rating
  - Years of Experience
  - Expertise

### 3. Prediction Button
- Provide a button to run the final Day 14 prediction models.
- Generate both EnrollmentCount and CourseRevenue predictions.

### 4. Enrollment KPI
- Display the predicted EnrollmentCount prominently.
- Present the value as the expected number of enrollments.

### 5. Revenue KPI
- Display the predicted CourseRevenue prominently.
- Present the value as the expected course revenue.

### 6. Demand Interpretation
- Convert predicted enrollment into a business-friendly demand level:
  - Below 150 → Low
  - 150–169 → Moderate
  - 170–184 → High
  - 185 or above → Very High

### 7. Business Recommendation
- Provide a recommendation based on the predicted demand level.
- Use the prediction to support course launch, pricing, or instructor planning decisions.

### 8. Prediction Input Summary
- Display the course information submitted by the user.
- Allow the user to review the inputs used for prediction.

### 9. Model Information
- Identify the final models used by the dashboard.
- Enrollment prediction: Day 14 Final Enrollment Model.
- Revenue prediction: Day 14 Final Revenue Model.
- Explain that the dashboard uses the saved models and does not retrain them.

## Day 16 Enhancement Over Day 15

Day 15 established the Streamlit prediction application.

Day 16 will transform that application into a business-oriented Demand Prediction Dashboard by adding:

- KPI-style prediction display
- Demand interpretation
- Business recommendation
- Prediction input summary
- Clear model information

## Verify the file from Jupyter

In [28]:
# ============================================================
# VERIFY DAY 16 APP FILE
# ============================================================

from pathlib import Path

app_path = Path(
    r"D:\Data Analytics Project\EduPro_Predictive_Modeling\app\app.py"
)

print("App path:")
print(app_path)

print("\nFile exists:")
print(app_path.exists())

print("\nFile size:")
print(app_path.stat().st_size, "bytes")

App path:
D:\Data Analytics Project\EduPro_Predictive_Modeling\app\app.py

File exists:
True

File size:
16620 bytes


## Verify the Day 16 dashboard components

In [29]:
# ============================================================
# VERIFY DAY 16 DASHBOARD COMPONENTS
# ============================================================

app_code = app_path.read_text(encoding="utf-8")

required_components = [
    "EduPro Demand Prediction Dashboard",
    "Predicted Enrollment",
    "Predicted Course Revenue",
    "Demand Level",
    "classify_demand",
    "generate_recommendation",
    "Prediction Input Summary",
    "Model Information",
    "EduPro_Final_Enrollment_Model.joblib",
    "EduPro_Final_Revenue_Model.joblib"
]

print("Day 16 Dashboard Component Validation")
print("=" * 50)

all_found = True

for component in required_components:

    found = component in app_code

    print(
        f"{component}: "
        f"{'FOUND' if found else 'MISSING'}"
    )

    if not found:
        all_found = False

print("=" * 50)

if all_found:
    print("SUCCESS: All required Day 16 components are present.")
else:
    print("WARNING: One or more components are missing.")

Day 16 Dashboard Component Validation
EduPro Demand Prediction Dashboard: FOUND
Predicted Enrollment: FOUND
Predicted Course Revenue: FOUND
Demand Level: FOUND
classify_demand: FOUND
generate_recommendation: FOUND
Prediction Input Summary: FOUND
Model Information: FOUND
EduPro_Final_Enrollment_Model.joblib: FOUND
EduPro_Final_Revenue_Model.joblib: FOUND
SUCCESS: All required Day 16 components are present.
